# Pairwise alignments

---
## Imports

In [1]:
import numpy as np

---
## Implement Smith-Waterman algorithm


In [2]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back

    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number

    Returns:
        score in position (i,j)
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT

    Pseudocode:
        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end

    '''
    # Get letters for the cell (-1 for extra border row / column)
    char1 = seq1[i-1]
    char2 = seq2[j-1]

    # Match or mismatch score for pairing the two letters
    if char1 == char2:
        pair_score = match
    else:
        pair_score = mismatch

    # Get scores from the three neighbors
    diag_score = matrix[i-1][j-1] + pair_score # upper-left 2letters
    up_score = matrix[i-1][j] + gap # up; seq1 letter against gap
    left_score = matrix[i][j-1] + gap # left: seq2 letter against gap

    # Best option
    score = max(0, diag_score, up_score, left_score)

    # Record direction the score came from
    if score == 0:
        move = 0 # END
    elif score == diag_score:
        move = 1 # DIAG
    elif score == up_score:
        move = 2 # UP
    else:
        move = 3 # LEFT

    return score, move



In [3]:
# # TRY OUT THE CODE ABOVE
import numpy as np
seq1 = "GATTC"
seq2 = "ATTG"
matrix = np.zeros((6,5))
matrix[1] = [0,0,0,0,2]
matrix[2] = [0,2,0,0,0]

print(cal_score(matrix, seq1, seq2, 3,2,2,-1,-2))

(np.float64(4.0), 1)


In [4]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''Find the optimal path through scoring marix

        diagonal: match/mismatch
        up: gap in seq1
        left: gap in seq2

    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from

    Returns:
        aligned_seq1 (str): e.g. GTTGAC
        aligned_seq2 (str): e.g. GTT-AC

    Pseudocode:
        while current_move != END:
            current_move = traceback_matrix[current_row][current_col]
            if current_move == DIAG:
                ...
            elif current_move == UP:
                ...
            elif current_move == LEFT:
                ...

    '''
    # Empty strings for sequence
    aligned_seq1 = ""
    aligned_seq2 = ""

    # Start at cell with highest score and read its arrow
    current_row, current_col  = maximum_position
    current_move = traceback_matrix[current_row][current_col]

    # Follow the arrows back until END (0)
    while current_move != 0:

        if current_move ==1:    # DIAG
            aligned_seq1 = seq1[current_row-1] + aligned_seq1
            aligned_seq2 = seq2[current_col-1] + aligned_seq2
            current_row = current_row - 1   # step up
            current_col = current_col - 1   # step left

        elif current_move == 2: # UP
            aligned_seq1 = seq1[current_row-1] + aligned_seq1
            aligned_seq2 = "-" + aligned_seq2
            current_row = current_row - 1   # step up

        elif current_move == 3: # LEFT
            aligned_seq1 = "-" + aligned_seq1
            aligned_seq2 = seq2[current_col -1] + aligned_seq2
            current_col = current_col - 1   # step left

        # Read the arrow in the new cell
        current_move = traceback_matrix[current_row][current_col]

    # Letters were added to the front, no reversing needed
    return aligned_seq1, aligned_seq2


In [5]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1):
    '''Smith-Waterman algorithm for local alignment

    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +1
        mismatch: default = -1
        gap: default = -1

    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix
    '''

    # Table size to include border row & column with 0s.
    rows = len(seq1) +1
    cols = len(seq2) +1

    # Score and arrow table filled with zeros to start
    score_matrix = np.zeros((rows, cols), dtype=int) # avoid the decimals/floats
    trace_matrix = np.zeros((rows, cols), dtype=int)

    # Trace highest score and its position
    max_score = 0
    max_pos = (0,0)

    # Fill in table row by row, left to right
    for i in range(1, rows):
        for j in range(1, cols):
            # Cell's scoreand arrow
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)
            score_matrix[i][j] = score
            trace_matrix[i][j] = move

            # Note cell if it has highest score sofar
            if score > max_score:
                max_score = score
                max_pos = (i, j)

    print("Alignment score:", max_score)

    # Follow arrows back from highest score to get the alignment
    aligned_seq1, aligned_seq2 = traceback(seq1, seq2, trace_matrix, max_pos)

    return aligned_seq1, aligned_seq2, score_matrix


In [6]:
# Example from slides
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1,seq2)

print (aligned_seq1)
print (aligned_seq2)
print (score_matrix)

Alignment score: 4
AC-TTA
ACATTA
[[0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 1 0 1 0 0 2 1]
 [0 1 0 2 1 0 0 1 1]
 [0 0 0 1 1 2 1 0 0]
 [0 0 0 0 0 2 3 2 1]
 [0 0 1 0 1 1 2 4 3]
 [0 0 0 0 0 0 1 3 3]]


In [7]:
seq3 = "GGGGAAAAAAAAAAAACCCC"
seq4 = "GGGGAAAAAACCCC"

aligned_seq3, aligned_seq4, score_matrix = smith_waterman(seq3,seq4)

print (aligned_seq3)
print (aligned_seq4)
print (score_matrix)


Alignment score: 10
GGGGAAAAAA
GGGGAAAAAA
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  1  1  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  2  2  2  1  0  0  0  0  0  0  0  0  0]
 [ 0  1  2  3  3  2  1  0  0  0  0  0  0  0  0]
 [ 0  1  2  3  4  3  2  1  0  0  0  0  0  0  0]
 [ 0  0  1  2  3  5  4  3  2  1  1  0  0  0  0]
 [ 0  0  0  1  2  4  6  5  4  3  2  1  0  0  0]
 [ 0  0  0  0  1  3  5  7  6  5  4  3  2  1  0]
 [ 0  0  0  0  0  2  4  6  8  7  6  5  4  3  2]
 [ 0  0  0  0  0  1  3  5  7  9  8  7  6  5  4]
 [ 0  0  0  0  0  1  2  4  6  8 10  9  8  7  6]
 [ 0  0  0  0  0  1  2  3  5  7  9  9  8  7  6]
 [ 0  0  0  0  0  1  2  3  4  6  8  8  8  7  6]
 [ 0  0  0  0  0  1  2  3  4  5  7  7  7  7  6]
 [ 0  0  0  0  0  1  2  3  4  5  6  6  6  6  6]
 [ 0  0  0  0  0  1  2  3  4  5  6  5  5  5  5]
 [ 0  0  0  0  0  1  2  3  4  5  6  5  4  4  4]
 [ 0  0  0  0  0  0  1  2  3  4  5  7  6  5  5]
 [ 0  0  0  0  0  0  0  1  2  3  4  6  8  7  6]
 [ 0  0  0  0  0  0  0  0  1  2  3  5  7  9  8